# FinanceMeta — Final NIFTY IV Reconstruction Benchmark

## Final result

The frozen 10%/20%/30% benchmark shows that **Linear Strike interpolation
has the lowest overall RMSE and MAE**, while SVI provides much higher
coverage at substantially higher runtime.

The temporal robustness result is more nuanced: SVI beats Linear in the
early evaluation regime, but loses to Linear in the late evaluation regime.

This notebook is for inspecting the final evidence. The protocol itself is
frozen in `docs/protocol.md`.

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "results").exists():
    ROOT = ROOT.parent

benchmark = pd.read_csv(ROOT / "results" / "benchmark_results.csv")
consistency = pd.read_csv(ROOT / "results" / "consistency_results.csv")
robustness = pd.read_csv(ROOT / "results" / "robustness_results.csv")

display(benchmark)

## Main benchmark comparison

In [ ]:
main = benchmark.copy()
main["missingness_pct"] = (main["missingness"] * 100).astype(int)

display(
    main[
        ["model", "missingness_pct", "rmse", "mae", "coverage",
         "runtime_seconds", "failure_count"]
    ].sort_values(["missingness_pct", "model"])
)

## What the main benchmark says

Linear Strike is the overall accuracy winner:

- 10%: RMSE 0.014114 vs SVI 0.024243
- 20%: RMSE 0.012079 vs SVI 0.039045
- 30%: RMSE 0.013816 vs SVI 0.038535

Nearest Strike is substantially worse on error at every missingness level.

The trade-off is coverage: Linear falls from 82.00% to 74.47%, while SVI
remains above 93%.

## SVI runtime and calibration failures

SVI runtime rises from about 37.8s to 54.4s while Linear remains below 0.7s.

SVI failures increase from 0 at 10% to 7 at 20% and 87 at 30%.

In [ ]:
svi = benchmark[benchmark["model"] == "svi"][[
    "missingness","runtime_seconds","svi_fit_failure_count","coverage","rmse","mae"
]]
display(svi)

## Temporal robustness

In [ ]:
robustness["missingness_pct"] = (robustness["missingness"] * 100).astype(int)
display(
    robustness[
        ["model","missingness_pct","regime","coverage","rmse","mae"]
    ].sort_values(["missingness_pct","regime","model"])
)

### Robustness interpretation

Early evaluation:

- 10%: SVI 0.000900 vs Linear 0.001478
- 20%: SVI 0.000950 vs Linear 0.001667
- 30%: SVI 0.001069 vs Linear 0.002508

Late evaluation:

- 10%: Linear 0.020661 vs SVI 0.035207
- 20%: Linear 0.017563 vs SVI 0.056254
- 30%: Linear 0.019716 vs SVI 0.055371

Therefore SVI's advantage is **regime-sensitive**, not stable across the full
evaluation period.

## Surface-consistency / arbitrage sanity

In [ ]:
consistency["missingness_pct"] = (consistency["missingness"] * 100).astype(int)
display(consistency)

These are basic call-price monotonicity and adjacent-strike convexity sanity
checks. They are not a proof of global arbitrage freedom. The dataset has one
expiry, so no calendar-spread test is claimed.

# Final conclusion

For this checkpoint, the evidence does **not** support replacing simple Linear
Strike interpolation with SVI on overall predictive accuracy.

The useful result is more nuanced:

> Linear is the strongest accuracy baseline; SVI trades much higher runtime
> for higher coverage and can outperform Linear in the earlier evaluation
> regime, but the advantage does not survive the later regime and SVI becomes
> increasingly fragile as missingness rises.

Next test: repeat the benchmark on a larger multi-expiry dataset with stronger
term-structure and no-arbitrage constraints.